<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_0_k_folds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_0_k_folds

## Introducción y Resumen

Esta notebook tiene como objetivo dividir el dataset MNQ en conjuntos de entrenamiento, validación y prueba, asegurando una partición aleatoria, reproducible y estructuralmente consistente. Con ello se dejan listos los datos para el entrenamiento y evaluación de los modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Carga de datos

    Se importa el dataset procesado con features técnicos y alpha factors. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Análisis del dataset `mnq_model`

    Se revisa la estructura del dataset (filas, columnas, tipos de datos), se busca los valores NaNs y se verifica la distribución temporal de los registros.

3. Definición de parámetros de división

    En este punto se define la estrategia de partición del dataset: se toma un 70% de los días para entrenamiento, y el 30% restante se divide en partes iguales para validación y prueba. De esta manera, el modelo cuenta con suficientes datos para aprender, mientras que se reservan bloques temporales separados para ajustar parámetros y evaluar el rendimiento final sin fugas de información.

4. Selección aleatoria de días.

    Este punto busca garantizar que la partición de los datos sea representativa y no esté sesgada por la secuencia temporal. Al asignar los días de forma aleatoria —aunque de manera reproducible— se evita que los conjuntos queden condicionados por períodos específicos del mercado (por ejemplo, tendencias prolongadas o alta volatilidad en ciertos meses). Así, cada subconjunto refleja mejor la diversidad del dataset y se obtiene una evaluación más robusta del modelo.

5. Generación de datasets `mnq_train`, `mnq_test` y `mnq_valid`

    En este punto se crean los datasets mnq_train, mnq_valid y mnq_test, manteniendo homogeneidad en estructura (301 registros por día, de 09:30 a 14:30) y sin solapamiento entre conjuntos. Esto asegura consistencia en el entrenamiento, validación y prueba del modelo.

## 0. Configuración del Entorno


### 0.1. Acceso a Drive

In [35]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [36]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [37]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr


## 1. Carga de datos

### 1.1. Carga de dataset `mqn_to_model`




In [66]:
def load_data(data: str):

    data_path = f'{drive_path}/5_transformer_90_model/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [67]:
mnq_model = load_data("to_model")

In [68]:
mnq_model

,date,open,high,low,close,volume,target_return_90,ire_60,momentum_5,rev_mom_vol_z_60,rev_mom_z_90,roc_60,rsi_14
datetime,,,,,,,,,,,,,
2020-01-02 04:30:00-05:00,2020-01-02,8813.25,8813.25,8812.50,8813.25,13,0.000539,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02 04:31:00-05:00,2020-01-02,8812.50,8812.50,8811.00,8812.50,121,0.000596,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02 04:32:00-05:00,2020-01-02,8812.50,8813.25,8811.75,8811.75,53,0.000539,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02 04:33:00-05:00,2020-01-02,8812.00,8812.25,8810.50,8810.50,37,0.000794,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02 04:34:00-05:00,2020-01-02,8810.75,8812.25,8810.50,8812.00,36,0.000709,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,NaN,-1.641428,-0.000947,0.686795,1.341640,-0.203125,42.096236
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,NaN,-1.553505,-0.000474,0.435459,0.998678,-0.182336,45.107585
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,NaN,-1.581640,-0.000035,0.231073,0.781652,-0.102800,44.313396


### 1.2. Información de dataset MNQ_to_model


In [69]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [70]:
num_dias, promedio_por_fecha = info_dataset(mnq_model, 'mnq_model')

Información del dataset mnq_model:

	Cantidad de días: 1295
	Registros por día: 691
	Hora diaria de inicio 04:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York



### 1.3. Carga de listado de features por ventana de tiempo

In [71]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/5_transformer_90_model/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]


In [72]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'rsi_14', 'momentum_5', 'rev_mom_vol_z_60']


In [73]:
features_base = ['open', 'high', 'low', 'close', 'volume']
features_90 = features_base + features_to_90

## 2. Análisis del dataset `mnq_model`

### 2.0. Funciones

#### Función para contar NaN en dataset

In [74]:
def nan_count(df):
  # Contar NaN por día y por columna
  daily_nan_counts = df.groupby("date").apply(lambda x: x.isna().sum())

  # Construir DataFrame con los valores únicos
  daily_unique_nans = pd.DataFrame({
      "feature": daily_nan_counts.columns,
      "daily_nan_counts": [sorted(daily_nan_counts[col].unique()) for col in daily_nan_counts.columns]
  })
  return daily_unique_nans

#### Función para detectar saltos temporales (gaps)

In [75]:
def detectar_gaps(df: pd.DataFrame, gap_minutes: int = 1):
    """
    Verifica si existen saltos mayores al intervalo esperado (por defecto 1 minuto)
    entre registros consecutivos dentro de cada día, en un DataFrame con índice tipo DatetimeIndex.

    Omite el primer registro de cada día.

    Parámetros:
    - df: DataFrame con índice datetime.
    - gap_minutes: tamaño esperado del intervalo en minutos (por defecto 1).

    Retorna:
    - Lista de índices donde se detectaron diferencias mayores al intervalo esperado.
    """
    df = df.copy()
    df['time_diff'] = df.index.to_series().diff()

    base_time_diff = pd.Timedelta(minutes=gap_minutes)
    problem_indices = []

    for date, group in df.groupby(df.index.date):
        time_diff = group['time_diff'].iloc[1:]
        incorrect_indices = time_diff[time_diff != base_time_diff].index
        if len(incorrect_indices) > 0:
            problem_indices.append(incorrect_indices)

    if problem_indices:
        print(f"Se encontraron problemas en {len(problem_indices)} registros con diferencias irregulares.\n")

        # Conteo por fecha
        conteos = df.groupby(df.index.date).size()

        for i in range(len(problem_indices)):
            idx = problem_indices[i][0]
            diff = df.loc[idx, 'time_diff']
            date = idx.date()
            count = conteos[date]
            print(f'\t{idx} -> Diferencia: {diff} | # Registros: {count}')
    else:
        print("No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.")

    return problem_indices

### 2.1. Búsqueda de NaN en `mnq_model`:

Buscamos los valores NaN en todas la columnas del dataset:

In [76]:
mnq_model_nans = nan_count(mnq_model)
mnq_model_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_90,[90]
7,ire_60,[270]
8,momentum_5,[5]
9,rev_mom_vol_z_60,[60]


Lo que se observa es:

- OHLCV y date → [0] → nunca tienen valores faltantes.

- Targets (target_return_*) → [30], [60], [90] → todos los días tienen esos NaN al final, consistente con la ventana de predicción que corta datos futuros.

- Factores técnicos → muchos muestran valores únicos iguales al tamaño de la ventana usada en su cálculo:

  - bb_60 → [59] → se necesitan 60 valores para calcular, por eso hay 59 NaN iniciales cada día.
  - ire_60, roc_60, rev_mom_vol_z_60 → [60].
  - ire_90, rev_mom_z_90 → [90].
  - roc_20 → [20].
  - momentum_5 → [5].

Caso particular:

- rev_score_90 → [1, 2, 3] → parece que en algunos días puede generar hasta 3 NaN, pero no es fijo como los demás.

Las ventanas más grandes con valores NaN son la `ire_90` y `rev_mom_z_90` que necesitan 90 minutos de historial.

Vamos a filtrar el dataset `mnq_model` para eliminar todos los NaNs:

In [77]:
# Filtrar filas sin NaN en ninguna columna
mnq_model_clean = mnq_model.dropna(how="any")

Verificamos si efectivamente no hay más NaNs en el dataset `mnq_model_clean`

In [78]:
mnq_model_clean_nans = nan_count(mnq_model_clean)
mnq_model_clean_nans

,feature,daily_nan_counts
0,date,[0]
1,open,[0]
2,high,[0]
3,low,[0]
4,close,[0]
5,volume,[0]
6,target_return_90,[0]
7,ire_60,[0]
8,momentum_5,[0]
9,rev_mom_vol_z_60,[0]


Se comprueba que no contamos con valores NaN. Ahora observemos la información del dataset:

In [79]:
num_dias, promedio_por_fecha = info_dataset(mnq_model_clean, 'mnq_model_clean')

Información del dataset mnq_model_clean:

	Cantidad de días: 1295
	Registros por día: 331
	Hora diaria de inicio 09:00
	Hora diaria de final 14:30
	Zona horaria: America/New_York



Como se observa en el resultado, el primer registro válido (sin valores NaN en ninguna columna) aparece a las 09:30 y la jornada finaliza a las 14:30.

A continuación, verificamos en el dataset filtrado si existen saltos en la secuencia temporal o si todos los registros se mantienen consecutivos.

### 2.2. Búsqueda de gaps en `mnq_model_clean`

In [80]:
detectar_gaps(mnq_model_clean)

No se encontraron problemas, todas las muestras son consecutivas minuto a minuto.


[]

Contamos con el dataset limpio de NaNs y saltos temporales.

In [81]:
mnq_model=mnq_model_clean.copy()

### 2.3. Filtramos los features correspondientes

In [82]:
mnq_model.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_90',
       'ire_60', 'momentum_5', 'rev_mom_vol_z_60', 'rev_mom_z_90', 'roc_60',
       'rsi_14'],
      dtype='object')

In [83]:
features_90

['open',
 'high',
 'low',
 'close',
 'volume',
 'ire_60',
 'rev_mom_z_90',
 'roc_60',
 'rsi_14',
 'momentum_5',
 'rev_mom_vol_z_60']

In [84]:
#Agregamos el features 'date' al inicio
features_90 = ['date'] + [col for col in features_90 if col != 'date']+['target_return_90']

In [85]:
mnq_90 = mnq_model[features_90].copy()

In [86]:
n_dias_90, promedio_por_fecha_90 = info_dataset(mnq_90, 'mnq_90')

Información del dataset mnq_90:

	Cantidad de días: 1295
	Registros por día: 331
	Hora diaria de inicio 09:00
	Hora diaria de final 14:30
	Zona horaria: America/New_York



### 2.3. Guardamos el dataset limpio

In [87]:
#Guardamos el dataset
ruta_mnq_90 = f'{drive_path}/5_transformer_90_model/mnq_90.parquet'
mnq_90.to_parquet(ruta_mnq_90, index=True)

In [88]:
mnq_90

,date,open,high,low,close,volume,ire_60,rev_mom_z_90,roc_60,rsi_14,momentum_5,rev_mom_vol_z_60,target_return_90
datetime,,,,,,,,,,,,,
2020-01-02 09:00:00-05:00,2020-01-02,8829.50,8829.50,8828.25,8828.25,61,-0.664949,-0.515742,0.110563,53.765816,-0.000255,-0.029341,-0.001190
2020-01-02 09:01:00-05:00,2020-01-02,8828.50,8829.50,8827.50,8828.00,98,-0.679484,-0.515773,0.104890,52.402631,-0.000227,-0.265588,-0.001218
2020-01-02 09:02:00-05:00,2020-01-02,8828.25,8828.25,8826.00,8826.00,139,-0.795764,-0.271922,0.068027,43.008129,-0.000396,-0.203084,-0.001644
2020-01-02 09:03:00-05:00,2020-01-02,8825.75,8826.00,8821.25,8822.00,625,-1.028324,0.032833,0.036853,31.027450,-0.000878,-0.941741,-0.000936
2020-01-02 09:04:00-05:00,2020-01-02,8822.00,8824.00,8822.00,8823.50,131,-0.941114,-0.007743,0.056699,38.002120,-0.000651,-0.012875,-0.000822
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,1036,-0.206521,1.460682,-0.357839,39.127063,-0.000299,0.514157,-0.004707
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,3542,-0.501943,1.855990,-0.419917,31.674514,-0.000817,1.892519,-0.003451
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,5241,-0.772747,2.120134,-0.493419,26.661409,-0.001416,3.140409,-0.002656


## 3. División por K-Folds

GroupKFold

Me permite dividir el dataset usando "grupos". Donde mis grupos son los días.

- No rompe la estructura temporal intradía
- No mezcla días entre folds
- Tenemos tantas divisiones como se necesiten (K=5, K=10, etc.)
- Es compatible con la arquitectura Transformer

Cuando usamos GroupKFold, los grupos son días completos.

Entonces:

- El K-Fold divide la lista de días, no los registros.
- Cada fold contiene un conjunto de días únicos, y ningún día se repite en otro fold cuando ese fold se usa como VALIDACIÓN.
- Los días se asignan a cada fold sin mezclar partes de días.
- Los días asignados a un fold no aparecen en el fold que se usa como validación en esa iteración.

OJO!!

🔹 Cada fold contiene días únicos dentro de ese fold?? Sí.

🔹 Pero esos mismos días vuelven a aparecer en otros folds como parte del train ?? También sí.




**Cada subgrupo contiene días únicos para esa partición del fold, pero esos mismos días pueden aparecer en otros folds cuando ese fold se usa como train en otra iteración.**

**Temporal K-Fold**

Los folds respetan el orden del tiempo. Nunca se usa un día futuro para entrenar un modelo que valida en días pasados.


Ejemplo con K=5:

- Fold 1:

    Train = Días 1–200

    Valid = Días 201–260

-  Fold 2:

    Train = Días 1–260

    Valid = Días 261–320

- Fold 3:

    Train = Días 1–320

    Valid = Días 321–381

→ siempre entrenás con el pasado, validás con un futuro que el modelo no ha visto.

Ventaja:

- No viola la naturaleza temporal del mercado.
- Evalúa cómo se comporta el modelo cuando avanza el tiempo.

Desventaja:

- Cada fold se parece más al anterior → menos diversidad que K-fold clásico.


La lógica es:

- Se ordenan los días en forma cronológica.
- Se reserva un bloque inicial de días como base mínima de entrenamiento (por defecto, 50% de los días).
- El resto de los días se divide en K bloques sucesivos de validación, siempre con:
  - Train = días anteriores
  - Valid = días posteriores (sin mezclar pasado/futuro)

En series temporales reales, el TEST siempre debe ser el bloque más reciente de días, porque simula el comportamiento del modelo “en el futuro”.

Por lo tanto:

- TEST = los últimos X días del dataset (por ejemplo 10%, 15% o una cantidad fija).

- TRAIN + VALID se obtienen con los Temporal K-Folds, pero sin tocar nunca el bloque de test.

In [89]:
import numpy as np
import pandas as pd

def temporal_kfold_con_test(
    df,
    date_col="date",
    K=5,
    test_frac=0.15,
    min_train_frac=0.70,
):
    """
    Crea K folds temporales + un conjunto de TEST separado (futuro).

    - TEST: últimos test_frac días.
    - TRAIN/VALID: K folds temporales con los días restantes.
    """

    # 1. Ordenar por fecha
    if "datetime" in df.columns:
        df = df.sort_values([date_col, "datetime"]).copy()
    else:
        df = df.sort_values(date_col).copy()

    # 2. Extraer días únicos
    unique_days = df[date_col].drop_duplicates().to_numpy()
    n_days = len(unique_days)

    # 3. Definir TEST: últimos test_frac días
    test_days = unique_days[int(n_days * (1 - test_frac)):]
    df_test = df[df[date_col].isin(test_days)].copy()

    # 4. Días disponibles para K-Fold
    remaining_days = unique_days[: int(n_days * (1 - test_frac))]
    n_rem = len(remaining_days)

    # 5. Base de entrenamiento mínima
    min_train_days = int(n_rem * min_train_frac)

    # 6. Días para validación en K bloques
    val_candidates = remaining_days[min_train_days:]
    n_val_candidates = len(val_candidates)
    val_size = n_val_candidates // K

    folds = []

    print(f"\n=== RESUMEN DEL DATASET ===")
    print(f"Total días: {n_days}")
    print(f"Días destinados a TEST: {len(test_days)}")
    print(f"Días restantes para K-Folds: {n_rem}")
    print("-" * 60)

    # Generar folds temporales
    for k in range(K):
        train_end_idx = min_train_days + k * val_size
        val_start_idx = train_end_idx

        if k < K - 1:
            val_end_idx = val_start_idx + val_size
        else:
            val_end_idx = n_rem  # último fold usa lo que queda

        train_days = remaining_days[:train_end_idx]
        val_days   = remaining_days[val_start_idx:val_end_idx]

        train_df = df[df[date_col].isin(train_days)].copy()
        val_df   = df[df[date_col].isin(val_days)].copy()

        folds.append({
            "fold": k + 1,
            "train_df": train_df,
            "val_df": val_df,
            "test_df": df_test,
            "train_days": train_days,
            "val_days": val_days,
            "test_days": test_days,
        })

        print(f"Fold {k+1}")
        print(f"  TRAIN: {len(train_days)} días | {len(train_df)} filas")
        print(f"    Fechas: {train_days[0]} → {train_days[-1]}")
        print(f"  VALID: {len(val_days)} días | {len(val_df)} filas")
        print(f"    Fechas: {val_days[0]} → {val_days[-1]}")
        print(f"  TEST (fijo): {len(test_days)} días | {len(df_test)} filas")
        print(f"    Fechas: {test_days[0]} → {test_days[-1]}")
        print("-" * 60)

    return folds

In [90]:
folds_final = temporal_kfold_con_test(
    mnq_90,
    date_col="date",
    K=5,             # o 10 si quieres
    test_frac=0.10, # últimos 10% días para test
    min_train_frac=0.50
)



=== RESUMEN DEL DATASET ===
Total días: 1295
Días destinados a TEST: 130
Días restantes para K-Folds: 1165
------------------------------------------------------------
Fold 1
  TRAIN: 582 días | 192642 filas
    Fechas: 2020-01-02 → 2022-06-16
  VALID: 116 días | 38396 filas
    Fechas: 2022-06-21 → 2022-12-12
  TEST (fijo): 130 días | 43030 filas
    Fechas: 2024-11-26 → 2025-06-13
------------------------------------------------------------
Fold 2
  TRAIN: 698 días | 231038 filas
    Fechas: 2020-01-02 → 2022-12-12
  VALID: 116 días | 38396 filas
    Fechas: 2022-12-14 → 2023-06-14
  TEST (fijo): 130 días | 43030 filas
    Fechas: 2024-11-26 → 2025-06-13
------------------------------------------------------------
Fold 3
  TRAIN: 814 días | 269434 filas
    Fechas: 2020-01-02 → 2023-06-14
  VALID: 116 días | 38396 filas
    Fechas: 2023-06-15 → 2023-12-05
  TEST (fijo): 130 días | 43030 filas
    Fechas: 2024-11-26 → 2025-06-13
-------------------------------------------------------

En cada fold:

- train_df → días históricos

- val_df → días posteriores a train, pero nunca mezcla futuro/pasado

- test_df → SIEMPRE los días más recientes (idéntico en todos los folds), representando el futuro real

Esto es exactamente lo que requiere un modelo temporal serio como nuestro Transformer.

El siguiente cuadro resume la cantidad de días asignados a **TRAIN**, **VALID** y **TEST** en cada fold:

| Fold | Train (días) | Valid (días) | Test (días) |
|------|--------------|--------------|-------------|
| 1    | 589          | 118          | 132         |
| 2    | 707          | 118          | 132         |
| 3    | 825          | 118          | 132         |
| 4    | 943          | 118          | 132         |
| 5    | 1061         | 118          | 132         |

**¿Por qué los TRAIN crecen en cada fold?**

Porque en un esquema **Temporal K-Fold**, los datos se respetan cronológicamente:

- Cada fold representa un momento distinto en el tiempo.
- TRAIN siempre incluye todo el pasado disponible hasta esa fecha.
- VALID es un bloque posterior no visto.
- TEST es el conjunto final, idéntico para todos los folds, simulando el **futuro real**.

**✔ Interpretación temporal de cada fold**

- **Fold 1** → Entrenar con información disponible hasta mediados de 2022  
- **Fold 2** → Entrenar con datos hasta fines de 2022 / inicios 2023  
- **Fold 3** → Entrenar con datos hasta mediados de 2023  
- **Fold 4** → Entrenar con datos hasta fines de 2023 / inicios 2024  
- **Fold 5** → Entrenar con prácticamente toda la historia disponible hasta 2024  

Y en **todos los folds**:

- **TEST** = últimos 132 días (2024/2025)

**✔ ¿Qué evalúa este esquema?**

Este método permite medir cómo se comporta el modelo **a medida que crece la historia disponible**, replicando escenarios reales de entrenamiento progresivo.


## 4. Guardado de Folds

In [91]:
drive_path

'/content/drive/MyDrive/neural_profit'

In [92]:
# Ruta base donde guardarás los folds
ruta_k_folds = f'{drive_path}/5_transformer_90_model/5_0_k_folds'
os.makedirs(ruta_k_folds, exist_ok=True)

In [93]:
for fold in folds_final:
    fold_id = fold["fold"]

    # Crear carpeta del fold
    fold_path = os.path.join(ruta_k_folds, f"fold_{fold_id}")
    os.makedirs(fold_path, exist_ok=True)

    # Extraer datasets
    train_df = fold["train_df"]
    val_df   = fold["val_df"]
    test_df  = fold["test_df"]   # mismo en todos los folds

    # Rutas de guardado
    train_path = os.path.join(fold_path, f"train_{fold_id}.parquet")
    val_path   = os.path.join(fold_path, f"valid_{fold_id}.parquet")
    test_path  = os.path.join(fold_path, f"test_{fold_id}.parquet")

    # Guardar archivos parquet
    train_df.to_parquet(train_path, index=True)
    val_df.to_parquet(val_path, index=True)
    test_df.to_parquet(test_path, index=True)

    print(f"Fold {fold_id} guardado:")
    print(f"  - TRAIN → {train_path}")
    print(f"  - VALID → {val_path}")
    print(f"  - TEST  → {test_path}")
    print("-" * 60)

Fold 1 guardado:
  - TRAIN → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_1/train_1.parquet
  - VALID → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_1/valid_1.parquet
  - TEST  → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_1/test_1.parquet
------------------------------------------------------------
Fold 2 guardado:
  - TRAIN → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_2/train_2.parquet
  - VALID → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_2/valid_2.parquet
  - TEST  → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_2/test_2.parquet
------------------------------------------------------------
Fold 3 guardado:
  - TRAIN → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_folds/fold_3/train_3.parquet
  - VALID → /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_0_k_fo